In [1]:
!pip install langchain faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 34.2 MB/s eta 0:00:00


In [8]:
!pip install -U langchain-community

In [7]:
from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.documents import Document

from datetime import datetime

In [9]:
documents = [

    {
        "text": "AI is transforming healthcare diagnostics",
        "source": "TechCrunch",
        "category": "technology",
        "date": "2024-02-10",
        "document_type": "technical"
    },

    {
        "text": "Top 10 AI startup trends this year",
        "source": "Medium",
        "category": "technology",
        "date": "2023-11-01",
        "document_type": "marketing"
    },

    {
        "text": "Football analytics uses machine learning",
        "source": "ESPN",
        "category": "sports",
        "date": "2024-01-15",
        "document_type": "technical"
    },

    {
        "text": "Healthy recipes improve nutrition",
        "source": "FoodBlog",
        "category": "cooking",
        "date": "2022-05-10",
        "document_type": "overview"
    },

    {
        "text": "Travel guides help tourists navigate cities",
        "source": "LonelyPlanet",
        "category": "travel",
        "date": "2024-03-01",
        "document_type": "overview"
    },

    {
        "text": "Cloud computing powers scalable AI systems",
        "source": "AWS Blog",
        "category": "technology",
        "date": "2024-04-15",
        "document_type": "technical"
    }
]

In [10]:
langchain_docs = []

for doc in documents:

    lc_doc = Document(
        page_content=doc["text"],
        metadata={
            "source": doc["source"],
            "category": doc["category"],
            "date": doc["date"],
            "document_type": doc["document_type"]
        }
    )

    langchain_docs.append(lc_doc)

print("Documents converted successfully!")

Documents converted successfully!


In [11]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_1087/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
vectorstore = FAISS.from_documents(
    langchain_docs,
    embedding_model
)

print("FAISS vector database created!")

FAISS vector database created!


In [13]:
def filtered_retrieve(query,
                      filters=None,
                      top_k=3):

    results = vectorstore.similarity_search_with_score(
        query,
        k=top_k
    )

    filtered_results = []

    for doc, score in results:

        metadata = doc.metadata

        include = True

        # CATEGORY FILTER
        if filters and "category" in filters:

            if metadata["category"] != filters["category"]:
                include = False

        # DATE FILTER
        if filters and "date_after" in filters:

            doc_date = datetime.strptime(
                metadata["date"],
                "%Y-%m-%d"
            )

            cutoff_date = datetime.strptime(
                filters["date_after"],
                "%Y-%m-%d"
            )

            if doc_date < cutoff_date:
                include = False

        if include:

            filtered_results.append({
                "text": doc.page_content,
                "metadata": metadata,
                "score": round(float(score), 4)
            })

    return filtered_results

In [14]:
queries = [

    "AI systems",
    "machine learning in sports",
    "travel navigation",
    "healthy food",
    "cloud infrastructure"
]

for query in queries:

    print("=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    results = filtered_retrieve(query)

    for r in results:
        print(r)

    print()

QUERY: AI systems
{'text': 'AI is transforming healthcare diagnostics', 'metadata': {'source': 'TechCrunch', 'category': 'technology', 'date': '2024-02-10', 'document_type': 'technical'}, 'score': 1.0407}
{'text': 'Cloud computing powers scalable AI systems', 'metadata': {'source': 'AWS Blog', 'category': 'technology', 'date': '2024-04-15', 'document_type': 'technical'}, 'score': 1.0579}
{'text': 'Top 10 AI startup trends this year', 'metadata': {'source': 'Medium', 'category': 'technology', 'date': '2023-11-01', 'document_type': 'marketing'}, 'score': 1.2357}

QUERY: machine learning in sports
{'text': 'Football analytics uses machine learning', 'metadata': {'source': 'ESPN', 'category': 'sports', 'date': '2024-01-15', 'document_type': 'technical'}, 'score': 0.4604}
{'text': 'AI is transforming healthcare diagnostics', 'metadata': {'source': 'TechCrunch', 'category': 'technology', 'date': '2024-02-10', 'document_type': 'technical'}, 'score': 1.567}
{'text': 'Top 10 AI startup trends t

In [15]:
filters = {
    "category": "technology",
    "date_after": "2023-01-01"
}

for query in queries:

    print("=" * 70)
    print("QUERY:", query)
    print("FILTERS:", filters)
    print("=" * 70)

    results = filtered_retrieve(
        query,
        filters=filters
    )

    for r in results:
        print(r)

    print()

QUERY: AI systems
FILTERS: {'category': 'technology', 'date_after': '2023-01-01'}
{'text': 'AI is transforming healthcare diagnostics', 'metadata': {'source': 'TechCrunch', 'category': 'technology', 'date': '2024-02-10', 'document_type': 'technical'}, 'score': 1.0407}
{'text': 'Cloud computing powers scalable AI systems', 'metadata': {'source': 'AWS Blog', 'category': 'technology', 'date': '2024-04-15', 'document_type': 'technical'}, 'score': 1.0579}
{'text': 'Top 10 AI startup trends this year', 'metadata': {'source': 'Medium', 'category': 'technology', 'date': '2023-11-01', 'document_type': 'marketing'}, 'score': 1.2357}

QUERY: machine learning in sports
FILTERS: {'category': 'technology', 'date_after': '2023-01-01'}
{'text': 'AI is transforming healthcare diagnostics', 'metadata': {'source': 'TechCrunch', 'category': 'technology', 'date': '2024-02-10', 'document_type': 'technical'}, 'score': 1.567}
{'text': 'Top 10 AI startup trends this year', 'metadata': {'source': 'Medium', 'cat

In [16]:
print("""

UPDATED RAG ARCHITECTURE

Raw Query
    ↓
Embedding Model
    ↓
Vector Similarity Search
    ↓
Metadata Filtering
    ↓
Filtered Relevant Documents
    ↓
Final Ranked Retrieval

""")



UPDATED RAG ARCHITECTURE

Raw Query
    ↓
Embedding Model
    ↓
Vector Similarity Search
    ↓
Metadata Filtering
    ↓
Filtered Relevant Documents
    ↓
Final Ranked Retrieval




In [17]:
print("""

EDGE CASES

1. Missing Metadata
Some documents may not contain category or date fields,
causing filters to fail.

2. Conflicting Categories
A document may belong to multiple categories
like both "technology" and "business",
making strict filtering unreliable.

""")



EDGE CASES

1. Missing Metadata
Some documents may not contain category or date fields,
causing filters to fail.

2. Conflicting Categories
A document may belong to multiple categories
like both "technology" and "business",
making strict filtering unreliable.




In [18]:
print("""

METADATA FILTERING IMPROVES PRECISION

Without filters:
- semantically similar but contextually wrong documents appear

With filters:
- retrieval becomes context-aware

Example:
A query about modern AI systems no longer retrieves outdated or unrelated content.

""")



METADATA FILTERING IMPROVES PRECISION

Without filters:
- semantically similar but contextually wrong documents appear

With filters:
- retrieval becomes context-aware

Example:
A query about modern AI systems no longer retrieves outdated or unrelated content.


